In [30]:
import pandas as pd
import ast
from seqeval.metrics.sequence_labeling import get_entities
import json
def extract_sc_labels(row):

    sc_label = ast.literal_eval(row["sc_label"])

    tokens = sc_label["text"].split()
    tags = sc_label["SC"]

    labels = []

    for category, start, end in get_entities(tags):
        labels.append({
            "span": " ".join(tokens[start:end+1]),
            "category": category
        })

    return {
        "id": int(row["Index_sentence"]),
        "text": " ".join(tokens),
        "labels": labels
    }

def extract_asqp_labels(row):

    label = ast.literal_eval(row["label"])

    tokens = label["Token"]

    quadruplets = []

    for _, start, end in get_entities(label["Span"]):

        category = label["Topic"][start]
        polarity = label["Sentiment"][start]

        # aspect
        aspect_entities = get_entities(
            label["Aspect"][start:end+1]
        )

        aspects = []


        for _, a_start, a_end in aspect_entities:
            aspects.append(
                " ".join(
                    tokens[start+a_start:start+a_end+1]
                )
            )

        aspect = (
            " | ".join(aspects)
            if aspects else "NULL"
        )

        # opinion
        opinion_tokens = [
            tokens[i]
            for i in range(start, end + 1)
            if label["Opinion"][i] == "1"
        ]

        opinion = (
            " ".join(opinion_tokens)
            if opinion_tokens else "NULL"
        )

        if opinion == "NULL" and aspect == "NULL":
            continue

        quadruplets.append({
            "category": category,
            "aspect": aspect,
            "opinion": opinion,
            "polarity": polarity
        })


    return {
        "id": int(row["Index_sentence"]),
        "text": " ".join(tokens),
        "labels": quadruplets
    }

In [49]:
split = "dev"
df = pd.read_csv(f"/kaggle/working/LLM-for-ABSA/data/raw/{split}.csv")

In [50]:
sc_output = df.apply(
    extract_sc_labels,
    axis=1
).tolist()

asqp_output = df.apply(
    extract_asqp_labels,
    axis=1
).tolist()

In [51]:
with open(rf"/kaggle/working/LLM-for-ABSA/data/ASQP/{split}.json", "w") as f:
    json.dump(asqp_output, f, indent=4, ensure_ascii=False)

In [52]:
with open(rf"/kaggle/working/LLM-for-ABSA/data/SACE/{split}.json", "w") as f:
    json.dump(asqp_output, f, indent=4, ensure_ascii=False)